# v14 画像的(非U-Net)TVT抽出法 10種 比較

v13 の PF 波形(`fwd_traj_*`/`bwd_traj_*`)と GR から **(H,T) 画像**(順方向密度・逆方向密度・GR尤度)を作り、
**学習なしの古典的「画像読み出し」**で各MD列の TVTオフセットを推定して比較する。

軸: y=中心(=順方向PF平均=center)からの相対TVT(ft), x=MD。予測 = 各列のオフセット → `pred_rel`。
評価: `target_rel`(=真TVT−center)基準の **overall RMSE**(全坑井プール) = TVT RMSE。
ベースライン: center そのもの(offset=0)。

手法(10):
1 順方向密度 argmax / 2 逆方向密度 argmax / 3 順×逆積 argmax / 4 順×逆積 soft-argmax
5 GR尤度 argmax / 6 順×逆×GR積 argmax / 7 同 soft-argmax
8 DP/Viterbi(融合コスト, 平滑最短経路) / 9 DP/Viterbi(GRコストのみ) / 10 積argmax→Savgol平滑

学習不要(教師なし読み出し)なので CV不要・overall RMSE を直接算出。

In [1]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.signal import savgol_filter
from numba import njit

In [2]:
# ============================== CONFIG ==============================
PROJECT = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
def _find(*c): return next((x for x in c if x.exists()), None)
RAW_DIR = _find(PROJECT/"v13"/"artifacts_v13"/"img", PROJECT/"artifacts_v13"/"img")
DATA_DIR = PROJECT / "rogii-wellbore-geology-prediction" / "train"
assert RAW_DIR is not None, "v13 img/ が無い"
print("RAW_DIR =", RAW_DIR)

BAND = 32.0; BINW = 0.5
H = int(2*BAND/BINW) + 1
OFF = (np.arange(H) - H//2) * BINW          # (H,) 相対TVT(ft)
KDE_SIG = 1.5                                # 密度KDE平滑(ビン)
DP_LAM = 0.3                                 # Viterbi 遷移ペナルティ(ビンあたり)
SAVGOL = (51, 2)                             # (窓, 次数)
N_WELLS = None

# GR尤度用 PF パラメータ(pf_1 の平滑/σ)
GRP = dict(HGR=3, TW=0, SIG_MIN=7.495879205722816, SIG_MAX=32.79346108797473,
           SIG_DEF=114.11439615641156, SIG_MULT=2.7482077023565075)
EPS = 1e-9

RAW_DIR = C:\Users\kosaka256\Documents\rogii_claude\v13\artifacts_v13\img


In [3]:
# ---- GR 材料(typewellから, PF不要) ----
def _smooth(vals, fb, r):
    r=int(r); s=pd.Series(vals,dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    return s.to_numpy(np.float32) if r<=0 else s.rolling(2*r+1,center=True,min_periods=1).mean().to_numpy(np.float32)
def _grid(tw_tvt, tw_gr, step=0.2):
    a,b=float(tw_tvt.min()),float(tw_tvt.max()); g=np.arange(a,b+step,step)
    return np.interp(g,tw_tvt,tw_gr).astype(np.float64), float(a), float(step)

def gr_lik(wid, center, ev_index):
    """各 (t, center+offset) の GR尤度 exp(-0.5*(resid/gs)^2) を (H,T) で返す。"""
    hw=pd.read_csv(DATA_DIR/f"{wid}__horizontal_well.csv")
    tw=pd.read_csv(DATA_DIR/f"{wid}__typewell.csv").sort_values("TVT")
    tw_tvt=tw["TVT"].to_numpy(float); tw_gr=tw["GR"].to_numpy(float)
    fb=float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    tw_gr_pf=_smooth(tw_gr,fb,GRP["TW"]).astype(np.float64)
    gg,gmin,gst=_grid(tw_tvt,tw_gr_pf); gg_axis=gmin+np.arange(len(gg))*gst
    hgr_known=_smooth(hw["GR"].astype(float).to_numpy(),fb,GRP["HGR"]); kn=hw[hw["TVT_input"].notna()]
    if len(kn)<20: gs=GRP["SIG_DEF"]
    else:
        kpos=hw.index.get_indexer(kn.index); resid=hgr_known[kpos]-np.interp(kn["TVT_input"].values,tw_tvt,tw_gr_pf)
        gs=float(np.nanstd(resid)); gs=GRP["SIG_DEF"] if (not np.isfinite(gs) or gs<=0) else gs
    gs=float(np.clip(gs*GRP["SIG_MULT"],GRP["SIG_MIN"],GRP["SIG_MAX"]))
    gr_full=hw["GR"].astype(float).interpolate(limit_direction="both").fillna(fb).to_numpy(float)
    gr_ev=_smooth(gr_full,fb,GRP["HGR"])[ev_index].astype(np.float64)
    T=len(center); tvt=center[:,None]+OFF[None,:]
    exp_gr=np.interp(tvt.ravel(),gg_axis,gg).reshape(T,H)
    lik=np.exp(-0.5*((gr_ev[:,None]-exp_gr)/gs)**2).T.astype(np.float32)   # (H,T)
    return lik

def traj_density(traj, center):
    T=traj.shape[1]; rel=traj.astype(np.float32)-center[None,:]
    idx=np.clip(np.round(rel/BINW).astype(np.int64)+H//2,0,H-1)
    dens=np.zeros((H,T),np.float32); tcol=np.arange(T)
    for s in range(traj.shape[0]): np.add.at(dens,(idx[s],tcol),np.float32(1.0))
    dens=gaussian_filter1d(dens,KDE_SIG,axis=0)
    c=dens.sum(0,keepdims=True); c[c<1e-12]=1.0
    return dens/c

def _collect(d, kind):
    keys=[k for k in d.files if k.startswith(f"{kind}_traj_")]
    if not keys: keys=[f"{kind}_traj"] if f"{kind}_traj" in d.files else []
    return np.concatenate([d[k].astype(np.float32) for k in keys],0)   # (sum S, T)

def well_maps(wid):
    d=np.load(RAW_DIR/f"{wid}.npz")
    if "target_rel" not in d.files: return None
    center=d["center"].astype(np.float64)
    hf=traj_density(_collect(d,"fwd"),center)
    hb=traj_density(_collect(d,"bwd"),center)
    hl=gr_lik(wid,center,d["ev_index"].astype(np.int64))
    return center, d["target_rel"].astype(np.float64), hf, hb, hl

In [4]:
# ---- 列正規化・argmax・soft-argmax・Viterbi(L1, numba) ----
def cnorm(m):
    s=m.sum(0,keepdims=True); s[s<EPS]=1.0; return m/s
def amax(m):   return OFF[m.argmax(0)]
def softmax_off(m):
    p=cnorm(np.clip(m,0,None)); return (p*OFF[:,None]).sum(0)

@njit(cache=False)
def viterbi_l1(cost, lam):
    Hh,T=cost.shape; INF=1e18
    dp=cost[:,0].copy(); back=np.zeros((Hh,T),np.int32)
    f=np.empty(Hh); arg=np.empty(Hh,np.int32)
    for t in range(1,T):
        for i in range(Hh): f[i]=dp[i]; arg[i]=i
        for i in range(1,Hh):
            if f[i-1]+lam<f[i]: f[i]=f[i-1]+lam; arg[i]=arg[i-1]
        for i in range(Hh-2,-1,-1):
            if f[i+1]+lam<f[i]: f[i]=f[i+1]+lam; arg[i]=arg[i+1]
        for j in range(Hh):
            dp[j]=cost[j,t]+f[j]; back[j,t]=arg[j]
    path=np.zeros(T,np.int32); b=0; bv=dp[0]
    for j in range(1,Hh):
        if dp[j]<bv: bv=dp[j]; b=j
    path[T-1]=b
    for t in range(T-1,0,-1): path[t-1]=back[path[t],t]
    return path

def dp_off(prob, lam):
    cost=-np.log(cnorm(np.clip(prob,0,None))+EPS).astype(np.float64)
    return OFF[viterbi_l1(np.ascontiguousarray(cost), float(lam))]

In [5]:
# ---- 10手法 ----
def methods(center, hf, hb, hl):
    fb = cnorm(hf)*cnorm(hb)
    allp = cnorm(hf)*cnorm(hb)*cnorm(hl)
    out = {}
    out["1 fwd argmax"]        = amax(hf)
    out["2 bwd argmax"]        = amax(hb)
    out["3 fb prod argmax"]    = amax(fb)
    out["4 fb prod softargmax"]= softmax_off(fb)
    out["5 gr argmax"]         = amax(hl)
    out["6 all prod argmax"]   = amax(allp)
    out["7 all prod softargmax"]= softmax_off(allp)
    out["8 DP all(viterbi)"]   = dp_off(allp, DP_LAM)
    out["9 DP gr(viterbi)"]    = dp_off(hl, DP_LAM)
    p10 = amax(fb); w = min(SAVGOL[0], (len(p10)//2)*2-1)
    out["10 fb argmax+savgol"] = savgol_filter(p10, w, SAVGOL[1]) if w>=5 else p10
    return out

In [6]:
wells = sorted(p.stem for p in RAW_DIR.glob("*.npz"))
if N_WELLS: wells = wells[:N_WELLS]
errs = None; base_err = []
t0=time.perf_counter()
for i,w in enumerate(wells):
    r = well_maps(w)
    if r is None: continue
    center, tr, hf, hb, hl = r
    base_err.append(0.0 - tr)                       # center(offset0) baseline
    preds = methods(center, hf, hb, hl)
    if errs is None: errs = {k: [] for k in preds}
    for k,off in preds.items():
        errs[k].append(off - tr)
    if (i+1)%100==0 or i+1==len(wells): print(f"  {i+1}/{len(wells)} ({time.perf_counter()-t0:.0f}s)")

  100/773 (14s)
  200/773 (27s)
  300/773 (40s)
  400/773 (57s)
  500/773 (70s)
  600/773 (84s)
  700/773 (97s)
  773/773 (108s)


In [7]:
def rmse(es):
    e=np.concatenate(es); e=e[np.isfinite(e)]; return float(np.sqrt(np.mean(e**2)))
base = rmse(base_err)
rows=[("0 center(baseline, offset0)", base)] + [(k, rmse(v)) for k,v in errs.items()]
df = pd.DataFrame(rows, columns=["method","overall_rmse"])
df["delta_vs_center"] = df["overall_rmse"] - base
df = df.sort_values("overall_rmse").reset_index(drop=True)
print("参考: v10(LGBM,2)=7.5537 / v12(TCN)=7.8710 / v13 U-Net(pf_1)=8.5829 / pf_ancc=9.1877")
df

参考: v10(LGBM,2)=7.5537 / v12(TCN)=7.8710 / v13 U-Net(pf_1)=8.5829 / pf_ancc=9.1877


,method,overall_rmse,delta_vs_center
0,"0 center(baseline, offset0)",9.276521,0.000000
1,4 fb prod softargmax,9.499061,0.222541
2,7 all prod softargmax,9.517197,0.240676
3,8 DP all(viterbi),11.340070,2.063549
4,1 fwd argmax,11.363710,2.087189
5,10 fb argmax+savgol,11.377304,2.100784
6,6 all prod argmax,11.551406,2.274885
7,3 fb prod argmax,11.552602,2.276081
8,9 DP gr(viterbi),12.965982,3.689461
9,2 bwd argmax,14.356840,5.080319
